# Notebook 03 -- Clustering Dual para Definición de Dominios de Expertos

---


## Objetivo

Definir dominios de expertos mediante clustering independiente en dos dimensiones: TYPE_DAY (perfiles diarios 24h) y TYPE_WEEK (perfiles semanales 168h). Cartesian product para asignación final.


## Entrada

data/processed/features.parquet


## Salida

Asignaciones de cluster por hora (type_day, type_week) + centroides de clusters


---
## Instalación

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

---
## Imports y Configuración

In [ ]:
import os
import json
import pickle
import warnings
from itertools import product
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

from tslearn.clustering import KShape, TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
CLUSTERING_DIR = os.path.join(DATA_DIR, "clustering")
RANDOM_SEED = 42
N_JOBS = -1

os.makedirs(CLUSTERING_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "serif",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = sns.color_palette("Set2", 10)

np.random.seed(RANDOM_SEED)
print("Configuración cargada.")
print(f"  DATA_DIR:       {os.path.abspath(DATA_DIR)}")
print(f"  CLUSTERING_DIR: {os.path.abspath(CLUSTERING_DIR)}")
print(f"  RANDOM_SEED:    {RANDOM_SEED}")

---
## Cargar datos

In [ ]:
features_path = os.path.join(DATA_DIR, "processed", "features.parquet")
df = pd.read_parquet(features_path)

# Asegurar índice datetime
if not isinstance(df.index, pd.DatetimeIndex):
    if "datetime" in df.columns:
        df = df.set_index("datetime")
    elif "fecha" in df.columns:
        df = df.set_index("fecha")
    else:
        # Usar primera columna si parece datetime
        first_col = df.columns[0]
        df[first_col] = pd.to_datetime(df[first_col])
        df = df.set_index(first_col)

df.index = pd.to_datetime(df.index)
df = df.sort_index()

# Identificar la columna objetivo (demand)
target_candidates = [c for c in df.columns if c.lower() in [
    "demanda", "demand", "demanda_sin", "target", "load", "value"
]]
if target_candidates:
    TARGET_COL = target_candidates[0]
else:
    # Respaldo: primera columna numérica
    TARGET_COL = df.select_dtypes(include=[np.number]).columns[0]


In [ ]:
print(f"Variables cargadas: {df.shape}")
print(f"Rango de fechas: {df.index.min()} a {df.index.max()}")
print(f"Columna objetivo: {TARGET_COL}")
print(f"Valores faltantes en el objetivo: {df[TARGET_COL].isna().sum()}")
df.head(3)

In [ ]:
start_date = '2022-01-01'
end_date = '2025-12-31'

df = df.loc[start_date:end_date]

print(f"DataFrame filtrado de {start_date} a {end_date}.")
print(f"Nueva forma del DataFrame: {df.shape}")
print(f"Rango de fechas: {df.index.min()} a {df.index.max()}")

---
## 1. Preparación de Perfiles

Construimos dos tipos de perfiles:

1. **Perfiles diarios (24D)**: Cada día se representa como un vector de 24 valores horarios de demanda.
2. **Perfiles semanales (168D)**: Cada semana ISO se representa como 168 valores horarios (Lun 00:00 a Dom 23:00).

Ambos se **normalizan Z por fila** para que el clustering capture la **forma** en lugar de la magnitud.
Esto es crítico porque queremos agrupar días/semanas con patrones similares, no con niveles absolutos
de demanda similares (que varían estacionalmente y con el crecimiento económico).

In [ ]:
# ── Perfiles diarios (vectores 24D) ──────────────────────────────────────────
demand_series = df[TARGET_COL].copy()

daily_profiles = demand_series.groupby(demand_series.index.date).apply(
    lambda x: x.values if len(x) == 24 else None
)
# Eliminar días con datos incompletos (!=24 horas)
daily_profiles = daily_profiles.dropna()
daily_dates = pd.to_datetime(daily_profiles.index)
daily_matrix = np.stack(daily_profiles.values)  # (n_días, 24)

print(f"Perfiles diarios: {daily_matrix.shape} (días x horas)")
print(f"  Días completos: {daily_matrix.shape[0]}")
print(f"  Días incompletos eliminados: {demand_series.index.normalize().nunique() - daily_matrix.shape[0]}")

#
daily_means = daily_matrix.mean(axis=1, keepdims=True)
daily_stds = daily_matrix.std(axis=1, keepdims=True)

# Evitar división por cero para días constantes (si los hay)
daily_stds[daily_stds < 1e-8] = 1.0
daily_normalized = (daily_matrix - daily_means) / daily_stds

print(f"\nForma de perfiles diarios normalizados: {daily_normalized.shape}")
print(f"  Medias por fila ~ 0: {np.abs(daily_normalized.mean(axis=1)).max():.2e}")
print(f"  Desviaciones estándar por fila ~ 1: {daily_normalized.std(axis=1).mean():.4f}")

In [ ]:
daily_matrix[0]

In [ ]:
daily_normalized[0]

In [ ]:
# ── Perfiles semanales (vectores 168D) ────────────────────────────────────────
# Agrupar por año-semana ISO, conservar solo semanas completas (168 horas)
demand_series_week = demand_series.copy()
demand_series_week = demand_series_week.to_frame(name="demand")
demand_series_week["iso_year"] = demand_series_week.index.isocalendar().year.values
demand_series_week["iso_week"] = demand_series_week.index.isocalendar().week.values
demand_series_week["year_week"] = (
    demand_series_week["iso_year"].astype(str) + "-W" +
    demand_series_week["iso_week"].astype(str).str.zfill(2)
)

weekly_groups = demand_series_week.groupby("year_week")["demand"]
weekly_profiles_dict = {}
weekly_start_dates = {}

for yw, group in weekly_groups:
    if len(group) == 168:
        weekly_profiles_dict[yw] = group.values
        weekly_start_dates[yw] = group.index.min()

weekly_labels = list(weekly_profiles_dict.keys())
weekly_matrix = np.stack(list(weekly_profiles_dict.values()))  # (n_semanas, 168)
weekly_dates = pd.to_datetime(list(weekly_start_dates.values()))

print(f"Perfiles semanales: {weekly_matrix.shape} (semanas x horas)")
print(f"  Semanas completas: {weekly_matrix.shape[0]}")

#
weekly_means = weekly_matrix.mean(axis=1, keepdims=True)
weekly_stds = weekly_matrix.std(axis=1, keepdims=True)
weekly_stds[weekly_stds < 1e-8] = 1.0
weekly_normalized = (weekly_matrix - weekly_means) / weekly_stds

print(f"\nForma de perfiles semanales normalizados: {weekly_normalized.shape}")
print(f"  Medias por fila ~ 0: {np.abs(weekly_normalized.mean(axis=1)).max():.2e}")
print(f"  Desviaciones estándar por fila ~ 1: {weekly_normalized.std(axis=1).mean():.4f}")

In [ ]:
weekly_matrix[0]

In [ ]:
weekly_normalized[0]

In [ ]:
# ── Comprobación visual rápida ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Ejemplo de perfiles diarios
ax = axes[0]
sample_idx = np.random.choice(len(daily_matrix), size=min(100, len(daily_matrix)), replace=False)
for i in sample_idx:
    ax.plot(range(24), daily_matrix[i], alpha=0.08, color="steelblue", linewidth=0.5)
ax.plot(range(24), daily_matrix.mean(axis=0), color="darkred", linewidth=2, label="Perfil promedio")
ax.set_xlabel("Hora del día")
ax.set_ylabel("Demanda")
ax.set_title("Perfiles diarios (24h)")
ax.set_xticks(range(0, 24, 3))
ax.legend()

# Ejemplo de perfiles semanales
ax = axes[1]
sample_idx_w = np.random.choice(len(weekly_matrix), size=min(50, len(weekly_matrix)), replace=False)
for i in sample_idx_w:
    ax.plot(range(168), weekly_matrix[i], alpha=0.08, color="steelblue", linewidth=0.5)
ax.plot(range(168), weekly_matrix.mean(axis=0), color="darkred", linewidth=2, label="Perfil promedio")
ax.set_xlabel("Hora de la semana (Lun 0h a Dom 23h)")
ax.set_ylabel("Demanda")
ax.set_title("Perfiles semanales (168h)")
day_ticks = [i * 24 for i in range(7)]
day_labels = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
ax.set_xticks(day_ticks)
ax.set_xticklabels(day_labels)
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "03_clustering_profiles_overview.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Preparación de perfiles completada.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sample daily profiles
ax = axes[0]
sample_idx = np.random.choice(len(daily_matrix), size=min(100, len(daily_matrix)), replace=False)
for i in sample_idx:
    ax.plot(range(24), daily_matrix[i], alpha=0.08, color="steelblue", linewidth=0.5)
ax.plot(range(24), daily_matrix.mean(axis=0), color="darkred", linewidth=2, label="Mean profile")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Demand")
ax.set_title("Daily Profiles (24h)")
ax.set_xticks(range(0, 24, 3))
ax.legend()

# Sample weekly profiles
ax = axes[1]
sample_idx_w = np.random.choice(len(weekly_matrix), size=min(50, len(weekly_matrix)), replace=False)
for i in sample_idx_w:
    ax.plot(range(168), weekly_matrix[i], alpha=0.08, color="steelblue", linewidth=0.5)
ax.plot(range(168), weekly_matrix.mean(axis=0), color="darkred", linewidth=2, label="Mean profile")
ax.set_xlabel("Hour of week (Mon 0h to Sun 23h)")
ax.set_ylabel("Demand")
ax.set_title("Weekly Profiles (168h)")
day_ticks = [i * 24 for i in range(7)]
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ax.set_xticks(day_ticks)
ax.set_xticklabels(day_labels)
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "english/03_clustering_profiles_overview.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Profile preparation complete.")

---
## 2. Clustering TYPE_DAY (Perfiles Diarios, 24h)

Comparamos cuatro algoritmos para k en {2, 3, 4, 5, 6, 7, 8} y evaluamos con
métricas de validación interna. La mejor configuración se selecciona mediante un
ranking combinado de Silhouette Score (mayor = mejor) y Davies-Bouldin Index (menor = mejor).

In [ ]:
def run_clustering(
    data_2d: np.ndarray,
    algorithm: str,
    k: int,
    random_seed: int = 42,
    max_iter: int = 50,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Ejecuta una configuración de clustering única.

    Parámetros
    ----------
    data_2d : np.ndarray de forma (n_muestras, n_variables)
        Perfiles normalizados Z.
    algorithm : str
        Uno de: 'kmeans_euclidean', 'kshape', 'kmeans_dtw', 'kmeans_softdtw'
    k : int
        Número de clusters.
    random_seed : int
        Semilla aleatoria para reproducibilidad.
    max_iter : int
        Número máximo de iteraciones para algoritmos tslearn.

    Retorna
    -------
    labels : np.ndarray de forma (n_muestras,)
    centroids : np.ndarray
    inertia : float
    """
    n_samples, n_features = data_2d.shape

    # tslearn requiere entrada 3D: (n_muestras, n_variables, 1)
    data_3d = data_2d.reshape(n_samples, n_features, 1)

    if algorithm == "kmeans_euclidean":
        model = TimeSeriesKMeans(
            n_clusters=k,
            random_state=random_seed,
            n_init=20,
            max_iter=max_iter,
            verbose=False,
            n_jobs=-1
        )
        labels = model.fit_predict(data_2d)
        centroids = model.cluster_centers_.squeeze()
        inertia = model.inertia_
        return labels, centroids, inertia

    if algorithm == "kshape":
        model = KShape(
            n_clusters=k,
            random_state=random_seed,
            max_iter=max_iter,
            verbose=False,
        )
        labels = model.fit_predict(data_2d)
        centroids = model.cluster_centers_.squeeze()  # (k, n_features)
        inertia = model.inertia_
        return labels, centroids, inertia

    elif algorithm == "kmeans_dtw":
        model = TimeSeriesKMeans(
            n_clusters=k,
            metric="dtw",
            random_state=random_seed,
            max_iter=max_iter,
            n_init=20,
            verbose=False,
            n_jobs=-1,
        )
        labels = model.fit_predict(data_2d)
        centroids = model.cluster_centers_.squeeze()
        inertia = model.inertia_
        return labels, centroids, inertia

    elif algorithm == "kmeans_softdtw":
        model = TimeSeriesKMeans(
            n_clusters=k,
            metric="softdtw",
            metric_params={"gamma": 0.1},
            random_state=random_seed,
            max_iter=max_iter,
            n_init=3,
            verbose=False,
        )
        labels = model.fit_predict(data_2d)
        centroids = model.cluster_centers_.squeeze()
        inertia = model.inertia_
        return labels, centroids, inertia

    else:
        raise ValueError(f"Algoritmo desconocido: {algorithm}")


def compute_clustering_metrics(
    data_2d: np.ndarray,
    labels: np.ndarray,
) -> Dict[str, float]:
    """
    Calcula métricas internas de validación de clustering.
    """
    n_unique = len(np.unique(labels))
    if n_unique < 2:
        return {
            "silhouette": -1.0,
            "davies_bouldin": np.inf,
            "calinski_harabasz": 0.0,
        }

    return {
        "silhouette": silhouette_score(data_2d, labels),
        "davies_bouldin": davies_bouldin_score(data_2d, labels),
        "calinski_harabasz": calinski_harabasz_score(data_2d, labels),
    }


print("Funciones auxiliares de clustering definidas.")

In [ ]:
# ── Barrido TYPE_DAY ────────────────────────────────────────────────────────
DAY_ALGORITHMS = ["kmeans_euclidean", "kshape", "kmeans_dtw"]
DAY_K_RANGE = list(range(2, 9))  # 2..8

day_results = []          # lista de diccionarios para la tabla de resultados
day_all_labels = {}       # (algo, k) -> labels
day_all_centroids = {}    # (algo, k) -> centroids

total_configs = len(DAY_ALGORITHMS) * len(DAY_K_RANGE)
print(f"Ejecutando {total_configs} configuraciones TYPE_DAY...\n")

for algo in DAY_ALGORITHMS:
    for k in DAY_K_RANGE:
        print(f"  {algo:20s} k={k} ...", end=" ", flush=True)
        try:
            labels, centroids, inertia = run_clustering(
                daily_matrix, algo, k, random_seed=RANDOM_SEED
            )
            metrics = compute_clustering_metrics(daily_matrix, labels)
            metrics["inertia"] = inertia
            metrics["algorithm"] = algo
            metrics["k"] = k
            metrics["clustering_type"] = "type_day"
            day_results.append(metrics)
            day_all_labels[(algo, k)] = labels
            day_all_centroids[(algo, k)] = centroids
            print(
                f"Sil={metrics['silhouette']:.4f}  "
                f"DB={metrics['davies_bouldin']:.4f}  "
                f"CH={metrics['calinski_harabasz']:.1f}"
            )
        except Exception as e:
            print(f"FALLÓ: {e}")
            day_results.append({
                "algorithm": algo, "k": k, "clustering_type": "type_day",
                "silhouette": np.nan, "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan, "inertia": np.nan,
            })

day_results_df = pd.DataFrame(day_results)
print(f"\nCompletadas {len(day_results)} configuraciones.")

In [ ]:
# ── Tabla de resultados TYPE_DAY ────────────────────────────────────────────────
display_cols = ["algorithm", "k", "silhouette", "davies_bouldin", "calinski_harabasz", "inertia"]
day_display = day_results_df[display_cols].copy()
day_display = day_display.sort_values(["algorithm", "k"]).reset_index(drop=True)

# Formato para visualización
styled = day_display.style.format({
    "silhouette": "{:.4f}",
    "davies_bouldin": "{:.4f}",
    "calinski_harabasz": "{:.1f}",
    "inertia": "{:.1f}",
}).background_gradient(
    subset=["silhouette"], cmap="Greens"
).background_gradient(
    subset=["davies_bouldin"], cmap="Reds_r"
)

print("Resultados de Clustering TYPE_DAY:")
display(styled)

In [ ]:
# ── Selección automática: ranking combinado por Silhouette + DB ───────────────
def rank_clustering_configs(results_df: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
    """
    Rankea configuraciones de clustering por Silhouette (mayor=mejor)
    y Davies-Bouldin (menor=mejor) siguiendo Chicco (2025).

    Retorna un DataFrame con el ranking, ordenado por ranking combinado.
    """
    df_rank = results_df.dropna(subset=["silhouette", "davies_bouldin"]).copy()

    # Ranking: silhouette descendente (rank 1 = mayor), DB ascendente (rank 1 = menor)
    df_rank["rank_silhouette"] = df_rank["silhouette"].rank(ascending=False).astype(int)
    df_rank["rank_db"] = df_rank["davies_bouldin"].rank(ascending=True).astype(int)
    df_rank["combined_rank"] = df_rank["rank_silhouette"] + df_rank["rank_db"]
    df_rank = df_rank.sort_values("combined_rank").reset_index(drop=True)

    return df_rank.head(top_n)


day_top3 = rank_clustering_configs(day_results_df, top_n=3)
print("TOP-3 configuraciones TYPE_DAY (por ranking combinado Silhouette + DB):")
print("=" * 80)
for i, row in day_top3.iterrows():
    print(
        f"  #{i+1}: {row['algorithm']:20s} k={int(row['k'])}  "
        f"Sil={row['silhouette']:.4f}  DB={row['davies_bouldin']:.4f}  "
        f"CH={row['calinski_harabasz']:.1f}  "
        f"(combined_rank={int(row['combined_rank'])})"
    )

# Mejor configuración
best_day_algo = day_top3.iloc[0]["algorithm"]
best_day_k = int(day_top3.iloc[0]["k"])
print(f"\nMejor TYPE_DAY: {best_day_algo} con k={best_day_k}")

---
## 3. Clustering TYPE_WEEK (Perfiles Semanales, 168h)

Mismo procedimiento que TYPE_DAY pero aplicado a los perfiles semanales 168h.

In [ ]:
# ── Barrido TYPE_WEEK ───────────────────────────────────────────────────────
WEEK_ALGORITHMS = ["kmeans_euclidean", "kshape", "kmeans_dtw"]
WEEK_K_RANGE = list(range(2, 6))  # 2..5

week_results = []
week_all_labels = {}
week_all_centroids = {}

total_configs_w = len(WEEK_ALGORITHMS) * len(WEEK_K_RANGE)
print(f"Ejecutando {total_configs_w} configuraciones TYPE_WEEK...\n")

for algo in WEEK_ALGORITHMS:
    for k in WEEK_K_RANGE:
        print(f"  {algo:20s} k={k} ...", end=" ", flush=True)
        try:
            labels, centroids, inertia = run_clustering(
                weekly_matrix, algo, k, random_seed=RANDOM_SEED
            )
            metrics = compute_clustering_metrics(weekly_matrix, labels)
            metrics["inertia"] = inertia
            metrics["algorithm"] = algo
            metrics["k"] = k
            metrics["clustering_type"] = "type_week"
            week_results.append(metrics)
            week_all_labels[(algo, k)] = labels
            week_all_centroids[(algo, k)] = centroids
            print(
                f"Sil={metrics['silhouette']:.4f}  "
                f"DB={metrics['davies_bouldin']:.4f}  "
                f"CH={metrics['calinski_harabasz']:.1f}"
            )
        except Exception as e:
            print(f"FALLÓ: {e}")
            week_results.append({
                "algorithm": algo, "k": k, "clustering_type": "type_week",
                "silhouette": np.nan, "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan, "inertia": np.nan,
            })

week_results_df = pd.DataFrame(week_results)
print(f"\nCompletadas {len(week_results)} configuraciones.")

In [ ]:
# ── Tabla de resultados TYPE_WEEK ───────────────────────────────────────────────
week_display = week_results_df[display_cols].copy()
week_display = week_display.sort_values(["algorithm", "k"]).reset_index(drop=True)

styled_w = week_display.style.format({
    "silhouette": "{:.4f}",
    "davies_bouldin": "{:.4f}",
    "calinski_harabasz": "{:.1f}",
    "inertia": "{:.1f}",
}).background_gradient(
    subset=["silhouette"], cmap="Greens"
).background_gradient(
    subset=["davies_bouldin"], cmap="Reds_r"
)

print("Resultados de Clustering TYPE_WEEK:")
display(styled_w)

In [ ]:
# ── Selección automática TYPE_WEEK ─────────────────────────────────────────
week_top3 = rank_clustering_configs(week_results_df, top_n=3)
print("TOP-3 configuraciones TYPE_WEEK (por ranking combinado Silhouette + DB):")
print("=" * 80)
for i, row in week_top3.iterrows():
    print(
        f"  #{i+1}: {row['algorithm']:20s} k={int(row['k'])}  "
        f"Sil={row['silhouette']:.4f}  DB={row['davies_bouldin']:.4f}  "
        f"CH={row['calinski_harabasz']:.1f}  "
        f"(combined_rank={int(row['combined_rank'])})"
    )

best_week_algo = week_top3.iloc[0]["algorithm"]
best_week_k = int(week_top3.iloc[0]["k"])
print(f"\nMejor TYPE_WEEK: {best_week_algo} con k={best_week_k}")

---
## 4. Combinación type_day x type_week

Cada observación horaria recibe:
- `cluster_day` de la mejor configuración TYPE_DAY
- `cluster_week` de la mejor configuración TYPE_WEEK
- `expert_id = f"day{cluster_day}_week{cluster_week}"`

El número total de expertos es **N = k_day x k_week**. Marcamos combinaciones raras
(<50 observaciones) que podrían requerir tratamiento especial (p. ej., fusión o agrupamiento).

In [ ]:
day_top3 = day_top3.sort_values("rank_db").reset_index(drop=True)

In [ ]:
best_day_k=3
best_week_k=2

In [ ]:
# ── Recuperar mejores etiquetas ──────────────────────────────────────────────────
best_day_labels = day_all_labels[(best_day_algo, best_day_k)]
best_week_labels = week_all_labels[(best_week_algo, best_week_k)]

# ── Construir DataFrame de asignación diaria ──────────────────────────────────────
day_assignment = pd.DataFrame({
    "date": daily_dates,
    "cluster_day": best_day_labels,
})
day_assignment["date"] = pd.to_datetime(day_assignment["date"])

# ── Construir DataFrame de asignación semanal ───────────────────────────────────
week_assignment = pd.DataFrame({
    "year_week": weekly_labels,
    "week_start": weekly_dates,
    "cluster_week": best_week_labels,
})

# ── Mapear asignaciones a cada observación horaria ──────────────────────────────
df_assignments = df[[TARGET_COL]].copy()
# Se conserva la fecha como columna: los merge de abajo descartan el índice y los
# notebooks siguientes unen las asignaciones a las variables por esta columna.
df_assignments["datetime"] = df_assignments.index
df_assignments["date"] = df_assignments.index.normalize()
df_assignments["iso_year"] = df_assignments.index.isocalendar().year.values
df_assignments["iso_week"] = df_assignments.index.isocalendar().week.values
df_assignments["year_week"] = (
    df_assignments["iso_year"].astype(str) + "-W" +
    df_assignments["iso_week"].astype(str).str.zfill(2)
)

# Merge de clusters diarios
df_assignments = df_assignments.merge(
    day_assignment, on="date", how="left"
)

# Merge de clusters semanales
df_assignments = df_assignments.merge(
    week_assignment[["year_week", "cluster_week"]], on="year_week", how="left"
)

# Eliminar observaciones sin asignación de cluster (días/semanas incompletas)
n_before = len(df_assignments)
df_assignments = df_assignments.dropna(subset=["cluster_day", "cluster_week"])
df_assignments["cluster_day"] = df_assignments["cluster_day"].astype(int)
df_assignments["cluster_week"] = df_assignments["cluster_week"].astype(int)
n_after = len(df_assignments)

print(f"Observaciones con ambas asignaciones: {n_after}/{n_before} ({n_after/n_before*100:.1f}%)")

# ── Crear expert_id ───────────────────────────────────────────────────────────
df_assignments["expert_id"] = (
    "day" + df_assignments["cluster_day"].astype(str) +
    "_week" + df_assignments["cluster_week"].astype(str)
)

n_experts = df_assignments["expert_id"].nunique()
print(f"\nTotal de expertos (k_day x k_week = {best_day_k} x {best_week_k}): {n_experts}")
print(f"\nFrecuencia por experto:")
print(df_assignments["expert_id"].value_counts().to_string())

In [ ]:
# ── Tabla de frecuencias: type_day x type_week ─────────────────────────────────
freq_table = pd.crosstab(
    df_assignments["cluster_day"],
    df_assignments["cluster_week"],
    margins=True,
    margins_name="Total",
)
freq_table.index.name = "cluster_day"
freq_table.columns.name = "cluster_week"

print("Tabla de frecuencias (observaciones horarias por type_day x type_week):")
display(freq_table)

# ── Identificar combinaciones raras ─────────────────────────────────────────
MIN_OBS_THRESHOLD = 50
expert_counts = df_assignments["expert_id"].value_counts()
rare_experts = expert_counts[expert_counts < MIN_OBS_THRESHOLD]

if len(rare_experts) > 0:
    print(f"\nCombinaciones raras (<{MIN_OBS_THRESHOLD} observaciones) — marcadas para tratamiento especial:")
    for eid, count in rare_experts.items():
        print(f"  {eid}: {count} observaciones")
else:
    print(f"\nNo hay combinaciones raras (<{MIN_OBS_THRESHOLD} obs). Todos los expertos tienen suficiente data.")

print(f"\nResumen:")
print(f"  Mejor configuración diaria:  {best_day_algo} k={best_day_k}")
print(f"  Mejor configuración semanal: {best_week_algo} k={best_week_k}")
print(f"  Total de expertos:           {n_experts}")
print(f"  Expertos raros:              {len(rare_experts)}")

---
## 5. Visualización

Gráficos de calidad de publicación para la tesis:
1. Perfil promedio de carga por cluster_day (curvas de 24h con bandas de confianza)
2. Perfil promedio de carga por cluster_week (curvas de 168h)
3. Heatmap de frecuencias type_day x type_week
4. Distribución temporal de las asignaciones de clusters (vista calendario)
5. Consistencia temporal: ARI en ventanas móviles trimestrales

In [ ]:
# ── 5.1 Perfil promedio diario por cluster_day ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

for c in sorted(np.unique(best_day_labels)):
    mask = best_day_labels == c
    cluster_profiles = daily_matrix[mask]
    mean_profile = cluster_profiles.mean(axis=0)
    std_profile = cluster_profiles.std(axis=0)

    color = PALETTE[c % len(PALETTE)]
    ax.plot(
        range(24), mean_profile, color=color, linewidth=2,
        label=f"Cluster {c} (n={mask.sum()})"
    )
    ax.fill_between(
        range(24),
        mean_profile - std_profile,
        mean_profile + std_profile,
        alpha=0.15, color=color,
    )

ax.set_xlabel("Hora del día")
ax.set_ylabel("Demanda (kWh)")
ax.set_title(
    f"Perfiles de Clusters TYPE_DAY ({best_day_algo}, k={best_day_k})"
)
ax.set_xticks(range(0, 24, 2))
ax.legend(loc="upper left", framealpha=0.9)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_day_profiles.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for c in sorted(np.unique(best_day_labels)):
    mask = best_day_labels == c
    cluster_profiles = daily_matrix[mask]
    mean_profile = cluster_profiles.mean(axis=0)
    std_profile = cluster_profiles.std(axis=0)

    color = PALETTE[c % len(PALETTE)]
    ax.plot(
        range(24), mean_profile, color=color, linewidth=2,
        label=f"Cluster {c} (n={mask.sum()})"
    )
    ax.fill_between(
        range(24),
        mean_profile - std_profile,
        mean_profile + std_profile,
        alpha=0.15, color=color,
    )

ax.set_xlabel("Hour of day")
ax.set_ylabel("Demand (kWh)")
ax.set_title(
    f"TYPE_DAY Cluster Profiles ({best_day_algo}, k={best_day_k})"
)
ax.set_xticks(range(0, 24, 2))
ax.legend(loc="upper left", framealpha=0.9)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "english/03_clustering_day_profiles.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# ── 5.2 Perfil promedio semanal por cluster_week ─────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for c in sorted(np.unique(best_week_labels)):
    mask = best_week_labels == c
    cluster_profiles = weekly_matrix[mask]
    mean_profile = cluster_profiles.mean(axis=0)
    std_profile = cluster_profiles.std(axis=0)

    color = PALETTE[c % len(PALETTE)]
    ax.plot(
        range(168), mean_profile, color=color, linewidth=1.5,
        label=f"Cluster {c} (n={mask.sum()})"
    )
    ax.fill_between(
        range(168),
        mean_profile - std_profile,
        mean_profile + std_profile,
        alpha=0.12, color=color,
    )

# Separadores de días
for d in range(1, 7):
    ax.axvline(d * 24, color="gray", linestyle=":", alpha=0.4, linewidth=0.8)

ax.set_xlabel("Hora de la semana")
ax.set_ylabel("Demanda (kWh)")
ax.set_title(
    f"Perfiles de Clusters TYPE_WEEK ({best_week_algo}, k={best_week_k})"
)
day_ticks = [i * 24 + 12 for i in range(7)]
day_labels = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
ax.set_xticks(day_ticks)
ax.set_xticklabels(day_labels)
ax.legend(loc="upper left", framealpha=0.9)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_week_profiles.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for c in sorted(np.unique(best_week_labels)):
    mask = best_week_labels == c
    cluster_profiles = weekly_matrix[mask]
    mean_profile = cluster_profiles.mean(axis=0)
    std_profile = cluster_profiles.std(axis=0)

    color = PALETTE[c % len(PALETTE)]
    ax.plot(
        range(168), mean_profile, color=color, linewidth=1.5,
        label=f"Cluster {c} (n={mask.sum()})"
    )
    ax.fill_between(
        range(168),
        mean_profile - std_profile,
        mean_profile + std_profile,
        alpha=0.12, color=color,
    )

# Day separators
for d in range(1, 7):
    ax.axvline(d * 24, color="gray", linestyle=":", alpha=0.4, linewidth=0.8)

ax.set_xlabel("Hour of week")
ax.set_ylabel("Demand (kWh)")
ax.set_title(
    f"TYPE_WEEK Cluster Profiles ({best_week_algo}, k={best_week_k})"
)
day_ticks = [i * 24 + 12 for i in range(7)]
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ax.set_xticks(day_ticks)
ax.set_xticklabels(day_labels)
ax.legend(loc="upper left", framealpha=0.9)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "english/03_clustering_week_profiles.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# ── 5.3 Heatmap de frecuencias type_day x type_week ──────────────────────
freq_no_margins = pd.crosstab(
    df_assignments["cluster_day"],
    df_assignments["cluster_week"],
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    freq_no_margins,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Número de observaciones horarias"},
)
ax.set_xlabel(f"cluster_week (k={best_week_k})")
ax.set_ylabel(f"cluster_day (k={best_day_k})")
ax.set_title("Frecuencia de Dominios de Expertos: type_day x type_week")

# Marcar combinaciones raras
for i in range(freq_no_margins.shape[0]):
    for j in range(freq_no_margins.shape[1]):
        val = freq_no_margins.iloc[i, j]
        if val < MIN_OBS_THRESHOLD:
            ax.add_patch(plt.Rectangle(
                (j, i), 1, 1, fill=False, edgecolor="red", linewidth=2.5
            ))

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_expert_heatmap.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Los bordes rojos indican combinaciones raras (<50 observaciones).")

In [ ]:
freq_no_margins = pd.crosstab(
    df_assignments["cluster_day"],
    df_assignments["cluster_week"],
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    freq_no_margins,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Number of hourly observations"},
)
ax.set_xlabel(f"cluster_week (k={best_week_k})")
ax.set_ylabel(f"cluster_day (k={best_day_k})")
ax.set_title("Expert Domain Frequency: type_day x type_week")

# Mark rare combinations
for i in range(freq_no_margins.shape[0]):
    for j in range(freq_no_margins.shape[1]):
        val = freq_no_margins.iloc[i, j]
        if val < MIN_OBS_THRESHOLD:
            ax.add_patch(plt.Rectangle(
                (j, i), 1, 1, fill=False, edgecolor="red", linewidth=2.5
            ))

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "english/03_clustering_expert_heatmap.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Red borders indicate rare combinations (<50 observations).")

In [ ]:
# ── 5.4 Distribución temporal de asignaciones cluster_day ──────────────────
day_assign_for_cal = day_assignment.copy()
day_assign_for_cal["year"] = day_assign_for_cal["date"].dt.year
years = sorted(day_assign_for_cal["year"].unique())

n_years = len(years)
fig, axes = plt.subplots(
    n_years, 1,
    figsize=(16, 1.8 * n_years + 1),
    squeeze=False,
)

cmap = plt.cm.get_cmap("Set2", best_day_k)

for idx, year in enumerate(years):
    ax = axes[idx, 0]
    year_data = day_assign_for_cal[day_assign_for_cal["year"] == year].copy()
    year_data["day_of_year"] = year_data["date"].dt.dayofyear
    year_data["week_of_year"] = year_data["date"].dt.isocalendar().week.values.astype(int)
    year_data["day_of_week"] = year_data["date"].dt.dayofweek  # 0=Lun

    # Crear matriz: semana x día_de_semana
    pivot = year_data.pivot_table(
        index="week_of_year", columns="day_of_week",
        values="cluster_day", aggfunc="first"
    )

    sns.heatmap(
        pivot.T,
        ax=ax,
        cmap=cmap,
        vmin=0, vmax=best_day_k - 1,
        linewidths=0.3,
        linecolor="white",
        cbar=idx == 0,
        cbar_kws={"label": "cluster_day", "ticks": range(best_day_k)} if idx == 0 else {},
        xticklabels=5,
        yticklabels=["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"],
    )
    ax.set_title(f"{year}", fontsize=11, fontweight="bold")
    ax.set_xlabel("" if idx < n_years - 1 else "Semana ISO")
    ax.set_ylabel("")

fig.suptitle(
    f"Distribución temporal de clusters TYPE_DAY ({best_day_algo}, k={best_day_k})",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_day_calendar.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
day_assign_for_cal = day_assignment.copy()
day_assign_for_cal["year"] = day_assign_for_cal["date"].dt.year
years = sorted(day_assign_for_cal["year"].unique())

n_years = len(years)
fig, axes = plt.subplots(
    n_years, 1,
    figsize=(16, 1.8 * n_years + 1),
    squeeze=False,
)

cmap = plt.cm.get_cmap("Set2", best_day_k)

for idx, year in enumerate(years):
    ax = axes[idx, 0]
    year_data = day_assign_for_cal[day_assign_for_cal["year"] == year].copy()
    year_data["day_of_year"] = year_data["date"].dt.dayofyear
    year_data["week_of_year"] = year_data["date"].dt.isocalendar().week.values.astype(int)
    year_data["day_of_week"] = year_data["date"].dt.dayofweek  # 0=Mon

    # Create a matrix: week x day_of_week
    pivot = year_data.pivot_table(
        index="week_of_year", columns="day_of_week",
        values="cluster_day", aggfunc="first"
    )

    sns.heatmap(
        pivot.T,
        ax=ax,
        cmap=cmap,
        vmin=0, vmax=best_day_k - 1,
        linewidths=0.3,
        linecolor="white",
        cbar=idx == 0,
        cbar_kws={"label": "cluster_day", "ticks": range(best_day_k)} if idx == 0 else {},
        xticklabels=5,
        yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    )
    ax.set_title(f"{year}", fontsize=11, fontweight="bold")
    ax.set_xlabel("" if idx < n_years - 1 else "ISO week")
    ax.set_ylabel("")

fig.suptitle(
    f"Temporal Distribution of TYPE_DAY Clusters ({best_day_algo}, k={best_day_k})",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "english/03_clustering_day_calendar.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# ── 5.5 Consistencia temporal: ARI en ventanas móviles trimestrales ────────────

def compute_sliding_ari(
    dates: np.ndarray,
    labels: np.ndarray,
    profiles: np.ndarray,
    algorithm: str,
    k: int,
    window_days: int = 90,
    step_days: int = 30,
    random_seed: int = 42,
) -> Tuple[List, List]:
    """
    Calcula ARI entre el clustering de todo el dataset y el re-clustering por ventana.
    Mide la estabilidad temporal de la estructura de clustering.
    """
    dates_pd = pd.to_datetime(dates)
    min_date = dates_pd.min()
    max_date = dates_pd.max()

    window_starts = []
    ari_values = []

    current_start = min_date
    while current_start + pd.Timedelta(days=window_days) <= max_date:
        window_end = current_start + pd.Timedelta(days=window_days)
        mask = (dates_pd >= current_start) & (dates_pd < window_end)

        if mask.sum() < k * 5:  # Necesita suficientes muestras
            current_start += pd.Timedelta(days=step_days)
            continue

        window_profiles = profiles[mask]
        window_labels_full = labels[mask]  # del clustering completo

        try:
            window_labels_new, _, _ = run_clustering(
                window_profiles, algorithm, k, random_seed=random_seed, max_iter=30
            )
            ari = adjusted_rand_score(window_labels_full, window_labels_new)
            window_starts.append(current_start + pd.Timedelta(days=window_days // 2))
            ari_values.append(ari)
        except Exception:
            pass

        current_start += pd.Timedelta(days=step_days)

    return window_starts, ari_values


# Calcular ARI para TYPE_DAY
print("Calculando consistencia temporal (ARI) para TYPE_DAY...")
day_ari_dates, day_ari_values = compute_sliding_ari(
    daily_dates, best_day_labels, daily_matrix,
    best_day_algo, best_day_k,
    window_days=90, step_days=30,
)

# Calcular ARI para TYPE_WEEK
print("Calculando consistencia temporal (ARI) para TYPE_WEEK...")
week_ari_dates, week_ari_values = compute_sliding_ari(
    weekly_dates, best_week_labels, weekly_matrix,
    best_week_algo, best_week_k,
    window_days=180, step_days=30,  # ventana mayor para semanal (menos muestras)
)

# Graficar
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

ax = axes[0]
ax.plot(day_ari_dates, day_ari_values, "o-", color="steelblue", markersize=4, linewidth=1.5)
ax.axhline(y=0.8, color="green", linestyle="--", alpha=0.5, label="ARI=0.8 (bueno)")
ax.axhline(y=0.5, color="orange", linestyle="--", alpha=0.5, label="ARI=0.5 (moderado)")
ax.set_ylabel("Adjusted Rand Index")
ax.set_title(
    f"Consistencia Temporal — TYPE_DAY ({best_day_algo}, k={best_day_k})\n"
    f"ARI entre clustering completo y ventana móvil de 90 días"
)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower left")

ax = axes[1]
ax.plot(week_ari_dates, week_ari_values, "s-", color="darkorange", markersize=4, linewidth=1.5)
ax.axhline(y=0.8, color="green", linestyle="--", alpha=0.5, label="ARI=0.8 (bueno)")
ax.axhline(y=0.5, color="orange", linestyle="--", alpha=0.5, label="ARI=0.5 (moderado)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Adjusted Rand Index")
ax.set_title(
    f"Consistencia Temporal — TYPE_WEEK ({best_week_algo}, k={best_week_k})\n"
    f"ARI entre clustering completo y ventana móvil de 180 días"
)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower left")

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_temporal_consistency.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

print(f"\nTYPE_DAY ARI:  media={np.mean(day_ari_values):.3f}, min={np.min(day_ari_values):.3f}")
print(f"TYPE_WEEK ARI: media={np.mean(week_ari_values):.3f}, min={np.min(week_ari_values):.3f}")

In [ ]:
def compute_sliding_ari(
    dates: np.ndarray,
    labels: np.ndarray,
    profiles: np.ndarray,
    algorithm: str,
    k: int,
    window_days: int = 90,
    step_days: int = 30,
    random_seed: int = 42,
) -> Tuple[List, List]:
    """
    Compute ARI between full-dataset clustering and per-window re-clustering.
    Measures temporal stability of the clustering structure.
    """
    dates_pd = pd.to_datetime(dates)
    min_date = dates_pd.min()
    max_date = dates_pd.max()

    window_starts = []
    ari_values = []

    current_start = min_date
    while current_start + pd.Timedelta(days=window_days) <= max_date:
        window_end = current_start + pd.Timedelta(days=window_days)
        mask = (dates_pd >= current_start) & (dates_pd < window_end)

        if mask.sum() < k * 5:  # Need enough samples
            current_start += pd.Timedelta(days=step_days)
            continue

        window_profiles = profiles[mask]
        window_labels_full = labels[mask]  # from full-dataset clustering

        try:
            window_labels_new, _, _ = run_clustering(
                window_profiles, algorithm, k, random_seed=random_seed, max_iter=30
            )
            ari = adjusted_rand_score(window_labels_full, window_labels_new)
            window_starts.append(current_start + pd.Timedelta(days=window_days // 2))
            ari_values.append(ari)
        except Exception:
            pass

        current_start += pd.Timedelta(days=step_days)

    return window_starts, ari_values


# Compute ARI for TYPE_DAY
print("Computing temporal consistency (ARI) for TYPE_DAY...")
day_ari_dates, day_ari_values = compute_sliding_ari(
    daily_dates, best_day_labels, daily_matrix,
    best_day_algo, best_day_k,
    window_days=90, step_days=30,
)

# Compute ARI for TYPE_WEEK
print("Computing temporal consistency (ARI) for TYPE_WEEK...")
week_ari_dates, week_ari_values = compute_sliding_ari(
    weekly_dates, best_week_labels, weekly_matrix,
    best_week_algo, best_week_k,
    window_days=180, step_days=30,  # larger window for weekly (fewer samples)
)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

ax = axes[0]
ax.plot(day_ari_dates, day_ari_values, "o-", color="steelblue", markersize=4, linewidth=1.5)
ax.axhline(y=0.8, color="green", linestyle="--", alpha=0.5, label="ARI=0.8 (good)")
ax.axhline(y=0.5, color="orange", linestyle="--", alpha=0.5, label="ARI=0.5 (moderate)")
ax.set_ylabel("Adjusted Rand Index")
ax.set_title(
    f"Temporal Consistency — TYPE_DAY ({best_day_algo}, k={best_day_k})\n"
    f"ARI between full-dataset and 90-day window re-clustering"
)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower left")

ax = axes[1]
ax.plot(week_ari_dates, week_ari_values, "s-", color="darkorange", markersize=4, linewidth=1.5)
ax.axhline(y=0.8, color="green", linestyle="--", alpha=0.5, label="ARI=0.8 (good)")
ax.axhline(y=0.5, color="orange", linestyle="--", alpha=0.5, label="ARI=0.5 (moderate)")
ax.set_xlabel("Date")
ax.set_ylabel("Adjusted Rand Index")
ax.set_title(
    f"Temporal Consistency — TYPE_WEEK ({best_week_algo}, k={best_week_k})\n"
    f"ARI between full-dataset and 180-day window re-clustering"
)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower left")

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "english/03_clustering_temporal_consistency.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

print(f"\nTYPE_DAY ARI:  mean={np.mean(day_ari_values):.3f}, min={np.min(day_ari_values):.3f}")
print(f"TYPE_WEEK ARI: mean={np.mean(week_ari_values):.3f}, min={np.min(week_ari_values):.3f}")

In [ ]:
# ── 5.6 Comparación de métricas ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Métricas TYPE_DAY
for algo in DAY_ALGORITHMS:
    subset = day_results_df[day_results_df["algorithm"] == algo]
    axes[0, 0].plot(subset["k"], subset["silhouette"], "o-", label=algo, markersize=5)
    axes[0, 1].plot(subset["k"], subset["davies_bouldin"], "o-", label=algo, markersize=5)

axes[0, 0].set_title("TYPE_DAY: Silhouette Score (mayor = mejor)")
axes[0, 0].set_xlabel("k")
axes[0, 0].set_ylabel("Silhouette")
axes[0, 0].legend(fontsize=8)

axes[0, 1].set_title("TYPE_DAY: Davies-Bouldin Index (menor = mejor)")
axes[0, 1].set_xlabel("k")
axes[0, 1].set_ylabel("DB Index")
axes[0, 1].legend(fontsize=8)

# Métricas TYPE_WEEK
for algo in WEEK_ALGORITHMS:
    subset = week_results_df[week_results_df["algorithm"] == algo]
    axes[1, 0].plot(subset["k"], subset["silhouette"], "o-", label=algo, markersize=5)
    axes[1, 1].plot(subset["k"], subset["davies_bouldin"], "o-", label=algo, markersize=5)

axes[1, 0].set_title("TYPE_WEEK: Silhouette Score (mayor = mejor)")
axes[1, 0].set_xlabel("k")
axes[1, 0].set_ylabel("Silhouette")
axes[1, 0].legend(fontsize=8)

axes[1, 1].set_title("TYPE_WEEK: Davies-Bouldin Index (menor = mejor)")
axes[1, 1].set_xlabel("k")
axes[1, 1].set_ylabel("DB Index")
axes[1, 1].legend(fontsize=8)

plt.suptitle("Métricas de validación interna por algoritmo y k", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_metrics_comparison.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# TYPE_DAY metrics
for algo in DAY_ALGORITHMS:
    subset = day_results_df[day_results_df["algorithm"] == algo]
    axes[0, 0].plot(subset["k"], subset["silhouette"], "o-", label=algo, markersize=5)
    axes[0, 1].plot(subset["k"], subset["davies_bouldin"], "o-", label=algo, markersize=5)

axes[0, 0].set_title("TYPE_DAY: Silhouette Score (higher = better)")
axes[0, 0].set_xlabel("k")
axes[0, 0].set_ylabel("Silhouette")
axes[0, 0].legend(fontsize=8)

axes[0, 1].set_title("TYPE_DAY: Davies-Bouldin Index (lower = better)")
axes[0, 1].set_xlabel("k")
axes[0, 1].set_ylabel("DB Index")
axes[0, 1].legend(fontsize=8)

# TYPE_WEEK metrics
for algo in WEEK_ALGORITHMS:
    subset = week_results_df[week_results_df["algorithm"] == algo]
    axes[1, 0].plot(subset["k"], subset["silhouette"], "o-", label=algo, markersize=5)
    axes[1, 1].plot(subset["k"], subset["davies_bouldin"], "o-", label=algo, markersize=5)

axes[1, 0].set_title("TYPE_WEEK: Silhouette Score (higher = better)")
axes[1, 0].set_xlabel("k")
axes[1, 0].set_ylabel("Silhouette")
axes[1, 0].legend(fontsize=8)

axes[1, 1].set_title("TYPE_WEEK: Davies-Bouldin Index (lower = better)")
axes[1, 1].set_xlabel("k")
axes[1, 1].set_ylabel("DB Index")
axes[1, 1].legend(fontsize=8)

plt.suptitle("Internal Validation Metrics by Algorithm and k", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "english/03_clustering_metrics_comparison.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

---
## 6. Guardar Archivos

Guardamos todos los artefactos de clustering para su uso posterior:

| Archivo | Descripción |
|---------|------------|
| `cluster_assignments.parquet` | cluster_day, cluster_week y expert_id por hora |
| `centroids_day.pkl` | Centroides TYPE_DAY (para asignación de nuevos datos) |
| `centroids_week.pkl` | Centroides TYPE_WEEK |
| `clustering_metrics.json` | Todas las métricas por (algoritmo, k) |
| `top_configs.json` | Top-3 configuraciones para evaluación posterior en NB08 |

In [ ]:
df_assignments.head(25)

In [ ]:
# ── 6.1 Guardar asignaciones de clusters ──────────────────────────────────────────
assignments_save = df_assignments[["datetime", "cluster_day", "cluster_week", "expert_id"]].copy()
assignments_path = os.path.join(CLUSTERING_DIR, "cluster_assignments.parquet")
assignments_save.to_parquet(assignments_path)
print(f"Guardado: {assignments_path}")
print(f"  Forma: {assignments_save.shape}")
print(f"  Columnas: {list(assignments_save.columns)}")

# ── 6.2 Guardar centroides ────────────────────────────────────────────────────
best_day_centroids = day_all_centroids[(best_day_algo, best_day_k)]
best_week_centroids = week_all_centroids[(best_week_algo, best_week_k)]

centroids_day_path = os.path.join(CLUSTERING_DIR, "centroids_day.pkl")
with open(centroids_day_path, "wb") as f:
    pickle.dump({
        "centroids": best_day_centroids,
        "algorithm": best_day_algo,
        "k": best_day_k,
        "normalization": "z_normalize_per_row",
    }, f)
print(f"\nGuardado: {centroids_day_path}")
print(f"  Forma de centroides: {best_day_centroids.shape}")

centroids_week_path = os.path.join(CLUSTERING_DIR, "centroids_week.pkl")
with open(centroids_week_path, "wb") as f:
    pickle.dump({
        "centroids": best_week_centroids,
        "algorithm": best_week_algo,
        "k": best_week_k,
        "normalization": "z_normalize_per_row",
    }, f)
print(f"\nGuardado: {centroids_week_path}")
print(f"  Forma de centroides: {best_week_centroids.shape}")

In [ ]:
# ── 6.3 Guardar todas las métricas de clustering ─────────────────────────────
all_metrics = pd.concat([day_results_df, week_results_df], ignore_index=True)
metrics_dict = all_metrics.to_dict(orient="records")

# Convertir tipos numpy a Python nativo para serialización JSON
def convert_numpy(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj) if not np.isnan(obj) else None
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

metrics_clean = []
for m in metrics_dict:
    metrics_clean.append({k: convert_numpy(v) for k, v in m.items()})

metrics_path = os.path.join(CLUSTERING_DIR, "clustering_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_clean, f, indent=2)
print(f"Guardado: {metrics_path}")
print(f"  Total de configuraciones: {len(metrics_clean)}")

# ── 6.4 Guardar top-3 configuraciones para evaluación posterior (NB08) ──────
top_configs = {
    "type_day": [],
    "type_week": [],
    "best_combination": {
        "day_algorithm": best_day_algo,
        "day_k": best_day_k,
        "week_algorithm": best_week_algo,
        "week_k": best_week_k,
        "n_experts": n_experts,
        "n_rare_experts": len(rare_experts),
    },
}

for _, row in day_top3.iterrows():
    top_configs["type_day"].append({
        "algorithm": row["algorithm"],
        "k": int(row["k"]),
        "silhouette": float(row["silhouette"]),
        "davies_bouldin": float(row["davies_bouldin"]),
        "calinski_harabasz": float(row["calinski_harabasz"]),
        "combined_rank": int(row["combined_rank"]),
    })

for _, row in week_top3.iterrows():
    top_configs["type_week"].append({
        "algorithm": row["algorithm"],
        "k": int(row["k"]),
        "silhouette": float(row["silhouette"]),
        "davies_bouldin": float(row["davies_bouldin"]),
        "calinski_harabasz": float(row["calinski_harabasz"]),
        "combined_rank": int(row["combined_rank"]),
    })

top_configs_path = os.path.join(CLUSTERING_DIR, "top_configs.json")
with open(top_configs_path, "w") as f:
    json.dump(top_configs, f, indent=2)
print(f"\nGuardado: {top_configs_path}")
print(f"  Top-3 configuraciones TYPE_DAY:  {[c['algorithm'] + '_k' + str(c['k']) for c in top_configs['type_day']]}")
print(f"  Top-3 configuraciones TYPE_WEEK: {[c['algorithm'] + '_k' + str(c['k']) for c in top_configs['type_week']]}")

In [ ]:
# ── 6.5 Guardar TODAS las etiquetas para las top-3 configs (para evaluación NB08) ─────
all_labels_archive = {
    "type_day": {},
    "type_week": {},
}

for _, row in day_top3.iterrows():
    key = f"{row['algorithm']}_k{int(row['k'])}"
    all_labels_archive["type_day"][key] = {
        "labels": day_all_labels[(row["algorithm"], int(row["k"]))].tolist(),
        "centroids": day_all_centroids[(row["algorithm"], int(row["k"]))].tolist(),
        "dates": [str(d) for d in daily_dates],
    }

for _, row in week_top3.iterrows():
    key = f"{row['algorithm']}_k{int(row['k'])}"
    all_labels_archive["type_week"][key] = {
        "labels": week_all_labels[(row["algorithm"], int(row["k"]))].tolist(),
        "centroids": week_all_centroids[(row["algorithm"], int(row["k"]))].tolist(),
        "year_weeks": weekly_labels,
    }

archive_path = os.path.join(CLUSTERING_DIR, "top3_labels_archive.pkl")
with open(archive_path, "wb") as f:
    pickle.dump(all_labels_archive, f)
print(f"Guardado: {archive_path}")
print(f"  Configuraciones TYPE_DAY:  {list(all_labels_archive['type_day'].keys())}")
print(f"  Configuraciones TYPE_WEEK: {list(all_labels_archive['type_week'].keys())}")

---
## Resumen

Este notebook completó el pipeline de clustering dual para la definición de dominios de expertos:

In [ ]:
print("=" * 80)
print("NOTEBOOK 03 — RESUMEN DEL CLUSTERING DUAL")
print("=" * 80)
print(f"\n1. PREPARACIÓN DE PERFILES")
print(f"   Perfiles diarios:  {daily_matrix.shape} (Z-normalizados por fila)")
print(f"   Perfiles semanales: {weekly_matrix.shape} (Z-normalizados por fila)")
print(f"\n2. CLUSTERING TYPE_DAY")
print(f"   Algoritmos probados: {DAY_ALGORITHMS}")
print(f"   Rango de k: {DAY_K_RANGE}")
print(f"   Mejor: {best_day_algo} con k={best_day_k}")
print(f"   Silhouette={day_top3.iloc[0]['silhouette']:.4f}, DB={day_top3.iloc[0]['davies_bouldin']:.4f}")
print(f"\n3. CLUSTERING TYPE_WEEK")
print(f"   Algoritmos probados: {WEEK_ALGORITHMS}")
print(f"   Rango de k: {WEEK_K_RANGE}")
print(f"   Mejor: {best_week_algo} con k={best_week_k}")
print(f"   Silhouette={week_top3.iloc[0]['silhouette']:.4f}, DB={week_top3.iloc[0]['davies_bouldin']:.4f}")
print(f"\n4. DOMINIOS DE EXPERTOS")
print(f"   N expertos = {best_day_k} x {best_week_k} = {n_experts}")
print(f"   Combinaciones raras (<{MIN_OBS_THRESHOLD} obs): {len(rare_experts)}")
print(f"   Total de observaciones horarias asignadas: {len(df_assignments)}")
print(f"\n5. CONSISTENCIA TEMPORAL")
if len(day_ari_values) > 0:
    print(f"   TYPE_DAY ARI:  media={np.mean(day_ari_values):.3f}, min={np.min(day_ari_values):.3f}")
if len(week_ari_values) > 0:
    print(f"   TYPE_WEEK ARI: media={np.mean(week_ari_values):.3f}, min={np.min(week_ari_values):.3f}")
print(f"\n6. ARTEFACTOS GUARDADOS")
print(f"   {assignments_path}")
print(f"   {centroids_day_path}")
print(f"   {centroids_week_path}")
print(f"   {metrics_path}")
print(f"   {top_configs_path}")
print(f"   {archive_path}")
print("\n" + "=" * 80)
print("Proceda al Notebook 04 para el entrenamiento de modelos por experto.")
print("Top-3 configuraciones guardadas para evaluación posterior en Notebook 08.")
print("=" * 80)